# SSTW S1 local Patch-pair dictionary screen

> **运行提示：选择 L4 GPU 后全部运行。** 本次只捕获真实 Wan step-4 OFF Q/K，并解析筛选冻结的 16 个局部单对 basis；不运行 VAE、MP4 或 S2。

METHOD_ONLY / DIAGNOSTIC_ONLY. The previous exact20 result remains S1_NO_GO_THIS_CONSTRUCTION; this screen tests whether a different frozen local Patch pair closes the relation-layer construction.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
REPOSITORY_URL = 'https://github.com/RICHAAARC/SC-SSTW-Feasibility.git'
AUTHORIZED_REF = '0000000000000000000000000000000000000000'
RUN_ID = '0000000000000000'
DRIVE_OUTPUT_ROOT = '/content/drive/MyDrive/SC-SSTW-Feasibility/s1-local-pair-dictionary'
AUTHORIZE_EXECUTION = False
AUTHORIZE_DRIVE_IO = False
MODEL_ID = 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers'
MODEL_REVISION = '0fad780a534b6463e45facd96134c9f345acfa5b'
EXPECTED_CONFIG_SHA256 = '8a717ec8afa6afcbe131fdc5a65c4cf6c4c6f0b1c03e2eae0b71582f69fefbdf'


In [ ]:
import subprocess
gpu_probe = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], check=False, capture_output=True, text=True)
print('GPU diagnostic:', gpu_probe.stdout.strip() or gpu_probe.stderr.strip() or 'unavailable')
print('L4 is supported; runtime acceptance is CUDA plus BF16 capability only.')


In [ ]:
from pathlib import Path
import hashlib, json, re, shutil, subprocess, sys, zipfile

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def require_absent(*paths):
    for path in paths:
        if path.exists() or path.is_symlink():
            raise RuntimeError(f'refusing to overwrite: {path}')

if not AUTHORIZE_EXECUTION or not AUTHORIZE_DRIVE_IO:
    raise RuntimeError('explicit execution and Drive acknowledgements are required')
if not re.fullmatch(r'[0-9a-f]{40}', AUTHORIZED_REF) or not re.fullmatch(r'[0-9a-f]{16}', RUN_ID):
    raise RuntimeError('exact ref and frozen run id are required')
WORK = Path('/content') / f'sstw-s1-pair-source-{RUN_ID}'
LOCAL_ROOT = Path('/content') / f'sstw-s1-pair-run-{RUN_ID}'
OUTPUT = LOCAL_ROOT / 'output'
LOG = Path('/content') / f'sstw-s1-pair-log-{RUN_ID}'
BUNDLE = Path('/content') / f'sstw-s1-pair-bundle-{RUN_ID}'
ARCHIVE = Path('/content') / f'sstw-s1-pair-dictionary-{RUN_ID}.zip'
SIDECAR = Path('/content') / f'sstw-s1-pair-dictionary-{RUN_ID}.zip.sha256.json'
DRIVE_ROOT = Path(DRIVE_OUTPUT_ROOT)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
if DRIVE_ROOT.is_symlink() or not DRIVE_ROOT.is_dir():
    raise RuntimeError('Drive output root is not a regular directory')
DRIVE_ARCHIVE = DRIVE_ROOT / ARCHIVE.name
DRIVE_SIDECAR = DRIVE_ROOT / SIDECAR.name
require_absent(WORK, LOCAL_ROOT, OUTPUT, LOG, BUNDLE, ARCHIVE, SIDECAR, DRIVE_ARCHIVE, DRIVE_SIDECAR)
LOG.mkdir()
runner_started = False
completed = None
audit = None
caught = None
try:
    with (LOG / 'clone.stdout').open('xb') as out, (LOG / 'clone.stderr').open('xb') as err:
        subprocess.run(['git', 'clone', '--no-checkout', REPOSITORY_URL, str(WORK)], check=True, stdout=out, stderr=err)
    with (LOG / 'checkout.stdout').open('xb') as out, (LOG / 'checkout.stderr').open('xb') as err:
        subprocess.run(['git', 'checkout', '--detach', AUTHORIZED_REF], cwd=WORK, check=True, stdout=out, stderr=err)
    head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=WORK, check=True, capture_output=True, text=True).stdout.strip()
    dirty = subprocess.run(['git', 'status', '--porcelain=v1', '--untracked-files=all'], cwd=WORK, check=True, capture_output=True, text=True).stdout
    if head != AUTHORIZED_REF or dirty:
        raise RuntimeError('clean exact-ref checkout failed')
    config_path = WORK / 'configs/s1_local_pair_dictionary.json'
    if sha256_file(config_path) != EXPECTED_CONFIG_SHA256:
        raise RuntimeError('frozen dictionary config mismatch')
    expected_run_id = hashlib.sha256(('SSTW-S1-LOCAL-PAIR-DICTIONARY:' + AUTHORIZED_REF + ':' + EXPECTED_CONFIG_SHA256).encode()).hexdigest()[:16]
    if RUN_ID != expected_run_id:
        raise RuntimeError('run id does not bind exact ref and config')
    locked = ['accelerate==1.4.0', 'diffusers==0.35.2', 'ftfy==6.3.1', 'huggingface_hub==0.35.3', 'numpy==1.26.4', 'safetensors==0.5.3', 'transformers==4.49.0']
    with (LOG / 'pip.stdout').open('xb') as out, (LOG / 'pip.stderr').open('xb') as err:
        subprocess.run([sys.executable, '-m', 'pip', 'install', *locked], check=True, stdout=out, stderr=err)
    import torch
    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
        raise RuntimeError('CUDA and BF16 capability are required')
    print('Runtime diagnostic:', {'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0)})
    from huggingface_hub import snapshot_download
    snapshot = Path(snapshot_download(repo_id=MODEL_ID, revision=MODEL_REVISION))
    if snapshot.is_symlink() or not snapshot.is_dir() or snapshot.resolve().name != MODEL_REVISION:
        raise RuntimeError('exact model snapshot identity failed')
    LOCAL_ROOT.mkdir()
    runner = WORK / 'experiments/run_s1_local_pair_dictionary.py'
    command = [sys.executable, str(runner), '--output', str(OUTPUT)]
    (LOG / 'command.json').write_text(json.dumps({'argv': command, 'cwd': str(WORK)}, sort_keys=True), encoding='utf-8')
    runner_started = True
    with (LOG / 'runner.stdout').open('xb') as out, (LOG / 'runner.stderr').open('xb') as err:
        completed = subprocess.run(command, cwd=WORK, stdout=out, stderr=err, check=False)
    runner_stdout = (LOG / 'runner.stdout').read_text(encoding='utf-8', errors='replace')
    runner_stderr = (LOG / 'runner.stderr').read_text(encoding='utf-8', errors='replace')
    print('Runner return code:', completed.returncode)
    if runner_stdout.strip(): print('Runner stdout:\n' + runner_stdout.rstrip())
    if runner_stderr.strip(): print('Runner stderr:\n' + runner_stderr.rstrip())
    if (OUTPUT / 'audit.json').is_file():
        audit = json.loads((OUTPUT / 'audit.json').read_text(encoding='utf-8'))
    expected_codes = {'PAIR_DICTIONARY_READY': 0, 'PAIR_DICTIONARY_NO_GO': 3}
    if audit is None or expected_codes.get(audit.get('status')) != completed.returncode:
        raise RuntimeError('runner audit missing or inconsistent; inspect stdout above')
    print('Dictionary status:', audit['status'])
except BaseException as exc:
    caught = exc
    print('Execution failure:', type(exc).__name__, str(exc))
finally:
    BUNDLE.mkdir()
    if LOG.exists(): shutil.copytree(LOG, BUNDLE / 'logs')
    if OUTPUT.exists() and OUTPUT.is_dir() and not OUTPUT.is_symlink(): shutil.copytree(OUTPUT, BUNDLE / 'output')
    state = {'schema': 'sstw.s1.local_pair_dictionary.notebook.v1', 'diagnostic_class': 'DIAGNOSTIC_ONLY', 'run_id': RUN_ID, 'source_ref': AUTHORIZED_REF, 'runner_started': runner_started, 'runner_return_code': None if completed is None else completed.returncode, 'audit_status': None if audit is None else audit.get('status'), 'failure_type': None if caught is None else type(caught).__name__, 'formal_result': False, 'stage_progression_allowed': False}
    (BUNDLE / 'notebook_state.json').write_text(json.dumps(state, sort_keys=True, separators=(',', ':')), encoding='utf-8')
    with zipfile.ZipFile(ARCHIVE, 'x', compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(BUNDLE.rglob('*')):
            if path.is_file(): archive.write(path, path.relative_to(BUNDLE))
    with zipfile.ZipFile(ARCHIVE, 'r') as archive:
        if archive.testzip() is not None: raise RuntimeError('local archive validation failed')
    archive_sha = sha256_file(ARCHIVE)
    sidecar = {'schema': 'sstw.s1.local_pair_dictionary.archive.v1', 'run_id': RUN_ID, 'source_ref': AUTHORIZED_REF, 'archive_name': ARCHIVE.name, 'archive_size': ARCHIVE.stat().st_size, 'archive_sha256': archive_sha, 'diagnostic_class': 'DIAGNOSTIC_ONLY'}
    SIDECAR.write_text(json.dumps(sidecar, sort_keys=True, separators=(',', ':')), encoding='utf-8')
    shutil.copyfile(ARCHIVE, DRIVE_ARCHIVE)
    shutil.copyfile(SIDECAR, DRIVE_SIDECAR)
    if DRIVE_ARCHIVE.stat().st_size != ARCHIVE.stat().st_size or sha256_file(DRIVE_ARCHIVE) != archive_sha:
        raise RuntimeError('Drive archive readback mismatch')
    if DRIVE_SIDECAR.read_text(encoding='utf-8') != SIDECAR.read_text(encoding='utf-8'):
        raise RuntimeError('Drive sidecar readback mismatch')
    with zipfile.ZipFile(DRIVE_ARCHIVE, 'r') as archive:
        if archive.testzip() is not None: raise RuntimeError('Drive archive validation failed')
    print('Packaged:', DRIVE_ARCHIVE, archive_sha)
if caught is not None:
    raise caught
